# 0 · Setup e carga dos dados
**Projeto:** Previsão de demanda — Corporación Favorita (Equador)

Neste notebook carregamos os dados brutos do Kaggle de forma otimizada para memória
e salvamos versões em **Parquet**, formato colunar muito mais rápido e leve que CSV.
Todos os notebooks seguintes partem desses arquivos.

## 1. Imports e caminhos
Importamos as bibliotecas e definimos onde estão os dados brutos (`data/raw`)
e onde salvaremos os dados tratados (`data/processed`).
O `print` final confirma que os CSVs foram encontrados.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

print(sorted(p.name for p in RAW.glob("*.csv")))

['holidays_events.csv', 'items.csv', 'oil.csv', 'sample_submission.csv', 'stores.csv', 'test.csv', 'train.csv', 'transactions.csv']


## 2. Carga otimizada do histórico de vendas (2017)
O `train.csv` tem ~125 milhões de linhas (~4,9 GB). Para caber na memória:
- **Leitura em blocos** (`chunksize`) de 5 milhões de linhas, filtrando cada bloco;
- **Apenas 2017** (jan–ago): período suficiente para treinar e prever as duas quinzenas;
- **Tipos enxutos**: `int8` para loja, `int32` para item, `float32` para vendas —
  reduzem o uso de memória em ~70% frente aos tipos padrão;
- **`onpromotion`** convertido de texto (`True`/`False`/vazio) para 1/0.

In [4]:
# Tipos de dados enxutos para economizar memória
dtypes = {"store_nbr": "int8", "item_nbr": "int32",
          "unit_sales": "float32", "onpromotion": "object"}

chunks = []
# Lê o arquivo em blocos de 5 milhões de linhas, ignorando a coluna 'id'
for chunk in pd.read_csv(RAW / "train.csv",
                         usecols=["date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"],
                         dtype=dtypes, parse_dates=["date"], chunksize=2_000_000):
    # Mantém só as vendas de 2017
    chunk = chunk[chunk["date"] >= "2017-01-01"].copy()
    if len(chunk):
        # Converte promoção para 1/0 (vazio vira 0)
        chunk["onpromotion"] = chunk["onpromotion"].map({"True": 1, "False": 0}).fillna(0).astype("int8")
        chunks.append(chunk)

# Junta os blocos em uma única tabela e libera a memória da lista
train = pd.concat(chunks, ignore_index=True)
del chunks

print(train.shape, f"{train.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(train["date"].min(), "->", train["date"].max())

(23808261, 5) 0.43 GB
2017-01-01 00:00:00 -> 2017-08-15 00:00:00


## 3. Tabelas auxiliares
Além das vendas, o dataset traz informações de contexto que vão virar *features* do modelo:

| Tabela | Conteúdo | Uso no projeto |
|---|---|---|
| `items` | família, classe e se o item é **perecível** | peso na métrica e análise de perdas |
| `stores` | cidade, estado, tipo e *cluster* da loja | comparação entre lojas e distribuição |
| `oil` | preço diário do petróleo | economia do Equador depende do petróleo |
| `holidays_events` | feriados nacionais, regionais e locais | picos e quedas de demanda |
| `transactions` | nº de transações por loja/dia (2013–2017) | tendência de longo prazo e fluxo de clientes |

In [5]:
# Carrega as tabelas de contexto (são pequenas, leitura direta)
items    = pd.read_csv(RAW / "items.csv")
stores   = pd.read_csv(RAW / "stores.csv")
oil      = pd.read_csv(RAW / "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(RAW / "holidays_events.csv", parse_dates=["date"])
trans    = pd.read_csv(RAW / "transactions.csv", parse_dates=["date"])

# Dicionário para facilitar loops de verificação e salvamento
aux = {"items": items, "stores": stores, "oil": oil,
       "holidays": holidays, "transactions": trans}

for nome, df in aux.items():
    print(f"{nome:13s} {df.shape}")

items         (4100, 4)
stores        (54, 5)
oil           (1218, 2)
holidays      (350, 6)
transactions  (83488, 3)


## 4. Limpeza básica e salvamento em Parquet
- **Vendas negativas** representam devoluções. Como o objetivo é prever *demanda*,
  elas são zeradas (`clip(lower=0)`).
- Os dados são salvos em **Parquet**: o arquivo fica até 10x menor que o CSV e
  carrega em segundos, preservando os tipos de dados.

In [6]:
# Devoluções aparecem como vendas negativas -> zeramos para modelar demanda
print("Linhas negativas (devoluções):", (train["unit_sales"] < 0).sum())
train["unit_sales"] = train["unit_sales"].clip(lower=0)

# Salva vendas e tabelas auxiliares em Parquet
train.to_parquet(PROC / "train_2017.parquet", index=False)
for nome, df in aux.items():
    df.to_parquet(PROC / f"{nome}.parquet", index=False)

print("Salvo! Arquivos em data/processed:")
print(sorted(p.name for p in PROC.glob("*.parquet")))

Linhas negativas (devoluções): 1633
Salvo! Arquivos em data/processed:
['holidays.parquet', 'items.parquet', 'oil.parquet', 'stores.parquet', 'train_2017.parquet', 'transactions.parquet']
